# InvestPy Incremental Local Rates Update
This notebook updates the existing local bond-yield files in `data/investpy_bond_yields` without doing a full re-download.

Default behavior: only refresh data from 2026 onward, using the existing `local_bond_download_summary.csv` as the update queue.

In [1]:
from __future__ import annotations

import time
from datetime import datetime
from pathlib import Path
from typing import Optional

import investpy
import pandas as pd
from IPython.display import display


def discover_project_root() -> Path:
    """Find the SeasonalityDashboard root from the current working directory."""
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data' / 'investpy_bond_yields').exists() and (candidate / 'app.py').exists():
            return candidate
    raise FileNotFoundError('Could not locate the SeasonalityDashboard project root.')


PROJECT_ROOT = discover_project_root()
LOCAL_BOND_DATA_DIR = PROJECT_ROOT / 'data' / 'investpy_bond_yields'
LOCAL_BOND_SUMMARY_PATH = LOCAL_BOND_DATA_DIR / 'local_bond_download_summary.csv'


def resolve_project_path(path_value: str | Path, project_root: Path = PROJECT_ROOT) -> Path:
    """Resolve stored summary paths across local and deployed environments."""
    path = Path(str(path_value))
    if path.exists():
        return path.resolve()

    if not path.is_absolute():
        return (project_root / path).resolve()

    parts = path.parts
    marker = ('data', 'investpy_bond_yields')
    for index in range(len(parts) - len(marker) + 1):
        if tuple(parts[index:index + len(marker)]) == marker:
            return project_root.joinpath(*marker, *parts[index + len(marker):]).resolve()

    return path.resolve()


def to_project_relative_path(path_value: str | Path, project_root: Path = PROJECT_ROOT) -> str:
    """Store data paths relative to the SeasonalityDashboard project root."""
    resolved_path = resolve_project_path(path_value, project_root=project_root)
    try:
        return resolved_path.relative_to(project_root).as_posix()
    except ValueError:
        return Path(path_value).as_posix()


print(f'Project root: {PROJECT_ROOT}')
print(f'Local bond data directory: {LOCAL_BOND_DATA_DIR}')
print(f'Summary file: {LOCAL_BOND_SUMMARY_PATH}')
print('Update mode: overwrite existing local rate files in place (no new app data structure).')

Project root: /Users/lijian/Desktop/NUSSIF/SeasonalityDashboard
Local bond data directory: /Users/lijian/Desktop/NUSSIF/SeasonalityDashboard/data/investpy_bond_yields
Summary file: /Users/lijian/Desktop/NUSSIF/SeasonalityDashboard/data/investpy_bond_yields/local_bond_download_summary.csv
Update mode: overwrite existing local rate files in place (no new app data structure).


In [2]:
def normalize_investpy_bond_frame(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Normalize InvestPy output to a yfinance-like OHLCV schema."""
    data = raw_df.copy()
    data.index = pd.to_datetime(data.index)
    data = data.sort_index()

    normalized_columns = {str(column).strip().lower(): str(column) for column in data.columns}
    rename_map: dict[str, str] = {}
    for canonical_name in ['open', 'high', 'low', 'close', 'volume']:
        if canonical_name in normalized_columns:
            rename_map[normalized_columns[canonical_name]] = canonical_name.title()

    data = data.rename(columns=rename_map)

    for required_column in ['Open', 'High', 'Low', 'Close', 'Volume']:
        if required_column not in data.columns:
            data[required_column] = pd.NA

    if 'Adj Close' not in data.columns:
        data['Adj Close'] = data['Close']

    return data[['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']]


def build_yearly_date_ranges(start_date: str | pd.Timestamp, end_date: str | pd.Timestamp) -> list[tuple[int, str, str]]:
    """Build partial-year / full-year ranges between two dates."""
    range_start = pd.Timestamp(start_date).normalize()
    range_end = pd.Timestamp(end_date).normalize()

    if range_start > range_end:
        return []

    ranges: list[tuple[int, str, str]] = []
    current_start = range_start

    while current_start <= range_end:
        current_year_end = pd.Timestamp(year=current_start.year, month=12, day=31)
        current_end = min(current_year_end, range_end)
        ranges.append((
            current_start.year,
            current_start.strftime('%d/%m/%Y'),
            current_end.strftime('%d/%m/%Y'),
        ))
        current_start = current_end + pd.Timedelta(days=1)

    return ranges


def download_investpy_bond_range(
    bond_name: str,
    from_date: str,
    to_date: str,
    retry_count: int = 60,
    retry_sleep_seconds: float = 1.0,
) -> pd.DataFrame:
    """Download one InvestPy date range with fixed retry spacing."""
    last_error = None

    for attempt in range(1, retry_count + 1):
        try:
            raw_df = investpy.get_bond_historical_data(
                bond=bond_name,
                from_date=from_date,
                to_date=to_date,
            )
            if raw_df is None or raw_df.empty:
                raise ValueError('InvestPy returned an empty DataFrame.')
            return normalize_investpy_bond_frame(raw_df)
        except Exception as exc:  # noqa: BLE001
            last_error = exc
            if attempt < retry_count:
                time.sleep(retry_sleep_seconds)

    raise RuntimeError(
        f'Failed to download {bond_name} from {from_date} to {to_date} after {retry_count} attempts: {last_error}'
    )


def load_local_bond_summary(summary_path: Path = LOCAL_BOND_SUMMARY_PATH) -> pd.DataFrame:
    """Load the existing local bond summary file."""
    if not summary_path.exists():
        raise FileNotFoundError(f'Local bond summary file not found: {summary_path}')

    summary_df = pd.read_csv(summary_path)
    if summary_df.empty:
        raise ValueError('Local bond summary is empty.')

    required_columns = {'country', 'bond_name', 'history_path', 'request_log_path'}
    missing_columns = required_columns - set(summary_df.columns)
    if missing_columns:
        raise ValueError(f'Local bond summary is missing columns: {sorted(missing_columns)}')

    summary_df = summary_df.copy()
    summary_df['history_path'] = summary_df['history_path'].map(
        lambda value: str(resolve_project_path(value))
    )
    summary_df['request_log_path'] = summary_df['request_log_path'].map(
        lambda value: str(resolve_project_path(value))
    )

    if 'status' in summary_df.columns:
        summary_df = summary_df[summary_df['status'].astype(str).str.lower() == 'ok'].copy()

    summary_df = summary_df[summary_df['history_path'].map(lambda value: Path(value).exists())].copy()
    return summary_df.sort_values(['country', 'bond_name']).reset_index(drop=True)

In [3]:
def build_update_queue(
    summary_df: pd.DataFrame,
    countries: Optional[list[str]] = None,
    bond_names: Optional[list[str]] = None,
    max_series: Optional[int] = None,
) -> pd.DataFrame:
    """Filter the local summary into an update queue."""
    queue_df = summary_df.copy()

    if countries:
        country_set = {str(value).strip().casefold() for value in countries}
        queue_df = queue_df[queue_df['country'].astype(str).str.casefold().isin(country_set)].copy()

    if bond_names:
        bond_set = {str(value).strip().casefold() for value in bond_names}
        queue_df = queue_df[queue_df['bond_name'].astype(str).str.casefold().isin(bond_set)].copy()

    if max_series is not None:
        queue_df = queue_df.head(int(max_series)).copy()

    return queue_df.reset_index(drop=True)


def update_single_local_bond(
    row: pd.Series,
    update_from_year: int = 2026,
    end_date: Optional[str] = None,
    retry_count: int = 60,
    retry_sleep_seconds: float = 1.0,
    inter_request_sleep_seconds: float = 1.0,
    overlap_buffer_days: int = 7,
) -> dict[str, object]:
    """Refresh one local bond file from max(update_from_year, recent overlap window) to today."""
    country = str(row['country'])
    bond_name = str(row['bond_name'])
    history_path = resolve_project_path(row['history_path'])
    request_log_path = resolve_project_path(row['request_log_path'])

    existing_history = pd.read_csv(history_path, index_col=0, parse_dates=True)
    if existing_history.empty:
        raise ValueError(f'Existing history is empty: {history_path}')

    existing_history.index = pd.to_datetime(existing_history.index, errors='coerce')
    existing_history = existing_history[~existing_history.index.isna()].sort_index()
    existing_history = existing_history[~existing_history.index.duplicated(keep='last')]

    if request_log_path.exists():
        existing_request_log = pd.read_csv(request_log_path)
    else:
        existing_request_log = pd.DataFrame()

    minimum_start = pd.Timestamp(year=int(update_from_year), month=1, day=1)
    final_end = pd.Timestamp(end_date or datetime.now().strftime('%Y-%m-%d')).normalize()
    last_saved_date = existing_history.index.max().normalize()
    refresh_start = max(minimum_start, last_saved_date - pd.Timedelta(days=int(overlap_buffer_days)))

    if refresh_start > final_end:
        return {
            'country': country,
            'bond_name': bond_name,
            'status': 'up_to_date',
            'rows_before': int(len(existing_history)),
            'rows_after': int(len(existing_history)),
            'new_rows_added': 0,
            'last_saved_before': last_saved_date.strftime('%Y-%m-%d'),
            'last_saved_after': last_saved_date.strftime('%Y-%m-%d'),
            'history_path': str(history_path),
            'request_log_path': str(request_log_path),
        }

    yearly_ranges = build_yearly_date_ranges(refresh_start, final_end)
    update_frames: list[pd.DataFrame] = []
    update_logs: list[dict[str, object]] = []

    for range_index, (year, from_date, to_date) in enumerate(yearly_ranges):
        print(f'Updating {country} | {bond_name} | {from_date} -> {to_date}')
        try:
            range_df = download_investpy_bond_range(
                bond_name=bond_name,
                from_date=from_date,
                to_date=to_date,
                retry_count=retry_count,
                retry_sleep_seconds=retry_sleep_seconds,
            )
            range_df['country'] = country
            range_df['bond_name'] = bond_name
            update_frames.append(range_df)
            update_logs.append({
                'country': country,
                'bond_name': bond_name,
                'year': year,
                'status': 'ok',
                'rows': int(len(range_df)),
                'from_date': from_date,
                'to_date': to_date,
                'update_run_timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            })
        except Exception as exc:  # noqa: BLE001
            update_logs.append({
                'country': country,
                'bond_name': bond_name,
                'year': year,
                'status': 'failed',
                'rows': 0,
                'from_date': from_date,
                'to_date': to_date,
                'error': str(exc),
                'update_run_timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            })
            print(f'Failed {country} | {bond_name} | {year}: {exc}')

        if range_index < len(yearly_ranges) - 1:
            time.sleep(inter_request_sleep_seconds)

    if not update_frames:
        update_log_df = pd.DataFrame(update_logs)
        combined_log = pd.concat([existing_request_log, update_log_df], axis=0, ignore_index=True)
        combined_log.to_csv(request_log_path, index=False)
        raise RuntimeError(
            f'No update data was downloaded for {country} | {bond_name}. Check {request_log_path}'
        )

    update_history = pd.concat(update_frames, axis=0)
    merged_history = pd.concat([existing_history, update_history], axis=0)
    merged_history = merged_history[~merged_history.index.duplicated(keep='last')].sort_index()

    update_log_df = pd.DataFrame(update_logs)
    combined_request_log = pd.concat([existing_request_log, update_log_df], axis=0, ignore_index=True)

    merged_history.to_csv(history_path)
    combined_request_log.to_csv(request_log_path, index=False)

    existing_dates = set(existing_history.index.astype(str))
    merged_dates = set(merged_history.index.astype(str))
    new_rows_added = len(merged_dates - existing_dates)
    last_saved_after = merged_history.index.max().normalize()
    status_label = 'updated' if last_saved_after > last_saved_date or new_rows_added > 0 else 'checked_no_new_rows'

    return {
        'country': country,
        'bond_name': bond_name,
        'status': status_label,
        'rows_before': int(len(existing_history)),
        'rows_after': int(len(merged_history)),
        'new_rows_added': int(new_rows_added),
        'last_saved_before': last_saved_date.strftime('%Y-%m-%d'),
        'last_saved_after': last_saved_after.strftime('%Y-%m-%d'),
        'history_path': str(history_path),
        'request_log_path': str(request_log_path),
    }


def run_incremental_update(
    update_queue: pd.DataFrame,
    summary_df: pd.DataFrame,
    update_from_year: int = 2026,
    end_date: Optional[str] = None,
    retry_count: int = 60,
    retry_sleep_seconds: float = 1.0,
    inter_request_sleep_seconds: float = 1.0,
    overlap_buffer_days: int = 7,
    summary_path: Path = LOCAL_BOND_SUMMARY_PATH,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Run an incremental refresh and overwrite the existing local files in place."""
    if update_queue.empty:
        raise ValueError('Update queue is empty. Adjust the filters and try again.')

    updated_summary_df = summary_df.copy()
    results: list[dict[str, object]] = []

    for row_number, (_, row) in enumerate(update_queue.iterrows(), start=1):
        print('=' * 110)
        print(f'[{row_number}/{len(update_queue)}] {row["country"]} | {row["bond_name"]}')
        try:
            result = update_single_local_bond(
                row=row,
                update_from_year=update_from_year,
                end_date=end_date,
                retry_count=retry_count,
                retry_sleep_seconds=retry_sleep_seconds,
                inter_request_sleep_seconds=inter_request_sleep_seconds,
                overlap_buffer_days=overlap_buffer_days,
            )
        except Exception as exc:  # noqa: BLE001
            result = {
                'country': str(row['country']),
                'bond_name': str(row['bond_name']),
                'status': 'failed',
                'rows_before': None,
                'rows_after': None,
                'new_rows_added': None,
                'last_saved_before': None,
                'last_saved_after': None,
                'history_path': str(resolve_project_path(row['history_path'])),
                'request_log_path': str(resolve_project_path(row['request_log_path'])),
                'error': str(exc),
            }
            print(f'Update failed for {row["country"]} | {row["bond_name"]}: {exc}')

        results.append(result)

        mask = (
            updated_summary_df['country'].astype(str).eq(str(row['country']))
            & updated_summary_df['bond_name'].astype(str).eq(str(row['bond_name']))
        )
        if mask.any() and result.get('status') in {'updated', 'checked_no_new_rows', 'up_to_date'}:
            history_df = pd.read_csv(str(result['history_path']), index_col=0, parse_dates=True)
            history_df.index = pd.to_datetime(history_df.index, errors='coerce')
            history_df = history_df[~history_df.index.isna()].sort_index()

            updated_summary_df.loc[mask, 'status'] = 'ok'
            updated_summary_df.loc[mask, 'rows'] = int(len(history_df))
            updated_summary_df.loc[mask, 'start_date'] = history_df.index.min().strftime('%Y-%m-%d')
            updated_summary_df.loc[mask, 'end_date'] = history_df.index.max().strftime('%Y-%m-%d')
            updated_summary_df.loc[mask, 'history_path'] = str(result['history_path'])
            updated_summary_df.loc[mask, 'request_log_path'] = str(result['request_log_path'])

    if 'history_path' in updated_summary_df.columns:
        updated_summary_df['history_path'] = updated_summary_df['history_path'].map(
            to_project_relative_path
        )

    if 'request_log_path' in updated_summary_df.columns:
        updated_summary_df['request_log_path'] = updated_summary_df['request_log_path'].map(
            to_project_relative_path
        )

    results_df = pd.DataFrame(results)
    updated_summary_df.to_csv(summary_path, index=False)

    print(f'Overwrote existing summary file: {summary_path}')
    print('Stored summary paths as project-relative paths for portability across environments.')
    print('No new app data files were created. The dashboard structure stays unchanged.')

    return results_df, updated_summary_df

In [4]:
UPDATE_FROM_YEAR = 2026
UPDATE_END_DATE = datetime.now().strftime('%Y-%m-%d')
UPDATE_RETRY_COUNT = 60
UPDATE_RETRY_SLEEP_SECONDS = 1.0
UPDATE_INTER_REQUEST_SLEEP_SECONDS = 1.0
UPDATE_OVERLAP_BUFFER_DAYS = 7
COUNTRY_FILTER = None  # Example: ['united states', 'japan']
BOND_NAME_FILTER = None  # Example: ['U.S. 10Y', 'Japan 10Y']
MAX_SERIES = None  # Example: 5
RUN_INCREMENTAL_UPDATE = False

local_bond_summary = load_local_bond_summary()
update_queue = build_update_queue(
    summary_df=local_bond_summary,
    countries=COUNTRY_FILTER,
    bond_names=BOND_NAME_FILTER,
    max_series=MAX_SERIES,
)

print(f'Total saved local series: {len(local_bond_summary)}')
print(f'Series queued for incremental update: {len(update_queue)}')
display(update_queue[['country', 'bond_name', 'start_date', 'end_date']].head(100))

if RUN_INCREMENTAL_UPDATE:
    incremental_update_results, refreshed_local_bond_summary = run_incremental_update(
        update_queue=update_queue,
        summary_df=local_bond_summary,
        update_from_year=UPDATE_FROM_YEAR,
        end_date=UPDATE_END_DATE,
        retry_count=UPDATE_RETRY_COUNT,
        retry_sleep_seconds=UPDATE_RETRY_SLEEP_SECONDS,
        inter_request_sleep_seconds=UPDATE_INTER_REQUEST_SLEEP_SECONDS,
        overlap_buffer_days=UPDATE_OVERLAP_BUFFER_DAYS,
    )
    display(incremental_update_results)
else:
    print('Incremental updater is ready.')
    print('Set RUN_INCREMENTAL_UPDATE = True and re-run this cell when you want to refresh local bond files.')
    print(f'Update start year floor: {UPDATE_FROM_YEAR}')
    print(f'Update end date: {UPDATE_END_DATE}')

Total saved local series: 112
Series queued for incremental update: 112


,country,bond_name,start_date,end_date
0,malaysia,Malaysia 3W,2006-08-22,2026-03-10
1,malaysia,Malaysia 3Y,2001-10-16,2026-03-11
2,malaysia,Malaysia 5Y,2001-10-17,2026-03-11
3,malaysia,Malaysia 7M,2006-08-22,2026-03-10
4,malaysia,Malaysia 7Y,2012-12-06,2026-03-11
...,...,...,...,...
95,united kingdom,U.K. 50Y,2005-05-19,2026-03-11
96,united kingdom,U.K. 5Y,2000-01-01,2026-03-11
97,united kingdom,U.K. 6M,2002-06-24,2026-03-11
98,united kingdom,U.K. 6Y,2000-01-03,2026-03-11


Incremental updater is ready.
Set RUN_INCREMENTAL_UPDATE = True and re-run this cell when you want to refresh local bond files.
Update start year floor: 2026
Update end date: 2026-03-12
